# Video 14: Filtering and Cleaning ChEMBL Data
Explanation Script:
"Now we filter the data to keep only high-quality records: IC50 values in nanomolar units, reasonable confidence, and remove duplicates. We'll display the cleaning progress step by step in tables."

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

# Load data
df = pd.read_csv('cox2_chembl_raw_data.csv')
print(f"Initial records: {len(df)}")

# Track cleaning progress
progress_data = []

# Step 1: Keep only IC50 measurements
df_ic50 = df[df['standard_type'] == 'IC50'].copy()
progress_data.append(['After IC50 filter', len(df_ic50), len(df) - len(df_ic50)])

# Step 2: Keep only nanomolar units
df_nm = df_ic50[df_ic50['standard_units'] == 'nM'].copy()
progress_data.append(['After nM filter', len(df_nm), len(df_ic50) - len(df_nm)])

# Step 3: Remove missing standard_values
df_clean = df_nm.dropna(subset=['standard_value']).copy()
progress_data.append(['After removing missing', len(df_clean), len(df_nm) - len(df_clean)])

# Step 4: Ensure numeric values
df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'], errors='coerce')
df_clean = df_clean.dropna(subset=['standard_value']).copy()
progress_data.append(['After numeric conversion', len(df_clean), 0])

# Step 5: Remove zero or negative values
df_clean = df_clean[df_clean['standard_value'] > 0].copy()
progress_data.append(['After removing <=0', len(df_clean), 0])

# Display cleaning progress
progress_df = pd.DataFrame(progress_data, columns=['Step', 'Records Remaining', 'Records Removed'])
print("\nCLEANING PROGRESS:")
display(progress_df)

# Step 6: Handle duplicates - take median for each molecule
print("\n" + "=" * 60)
print("DUPLICATE HANDLING")
print("=" * 60)

# Show duplicates before handling
duplicate_counts = df_clean['molecule_chembl_id'].value_counts()
molecules_with_duplicates = duplicate_counts[duplicate_counts > 1]
print(f"Molecules with multiple measurements: {len(molecules_with_duplicates)}")
print(f"Average measurements per molecule: {duplicate_counts.mean():.1f}")
print("\nMolecules with most measurements:")
display(molecules_with_duplicates.head(10).to_frame('Count'))

# Take median for each molecule
df_median = df_clean.groupby('molecule_chembl_id').agg({
    'standard_value': 'median',
    'canonical_smiles': 'first',
    'document_year': 'min'
}).reset_index()

print(f"\nAfter duplicate handling: {len(df_median)} unique molecules")

# Show sample of cleaned data
print("\nCLEANED DATA SAMPLE (first 10 molecules):")
display(df_median.head(10))

# Summary statistics of cleaned data
print("\nCLEANED DATA STATISTICS:")
stats_df = pd.DataFrame({
    'Metric': ['Total Molecules', 'Mean IC50 (nM)', 'Median IC50 (nM)', 'Min IC50 (nM)', 'Max IC50 (nM)'],
    'Value': [
        f"{len(df_median)}",
        f"{df_median['standard_value'].mean():.1f}",
        f"{df_median['standard_value'].median():.1f}",
        f"{df_median['standard_value'].min():.1f}",
        f"{df_median['standard_value'].max():.1f}"
    ]
})
display(stats_df)

# Save cleaned data
df_median.to_csv('cox2_chembl_cleaned.csv', index=False)
print(f"\n✓ Cleaned data saved to 'cox2_chembl_cleaned.csv'")

Initial records: 8383

CLEANING PROGRESS:


,Step,Records Remaining,Records Removed
0,After IC50 filter,8383,0
1,After nM filter,7372,1011
2,After removing missing,7353,19
3,After numeric conversion,7353,0
4,After removing <=0,7346,0



DUPLICATE HANDLING
Molecules with multiple measurements: 1016
Average measurements per molecule: 1.4

Molecules with most measurements:


,Count
molecule_chembl_id,
CHEMBL118,226
CHEMBL6,127
CHEMBL122,64
CHEMBL521,36
CHEMBL139,28
CHEMBL42485,25
CHEMBL7162,23
CHEMBL25,20
CHEMBL416146,16



After duplicate handling: 5371 unique molecules

CLEANED DATA SAMPLE (first 10 molecules):


,molecule_chembl_id,standard_value,canonical_smiles,document_year
0,CHEMBL100091,60.0,CCCCOCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(...,2000.0
1,CHEMBL100092,60.0,CCCCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(O...,2000.0
2,CHEMBL100156,4000.0,CS(=O)(=O)c1ccc(-c2cscc2-c2ccccc2)cc1,1996.0
3,CHEMBL100250,4000.0,CS(=O)(=O)c1ccc(-c2ccsc2-c2ccccc2)cc1,1996.0
4,CHEMBL100331,40.0,CCCCCCCCNC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc...,2000.0
5,CHEMBL10042,33000.0,CS(=O)(=O)c1ccc(-c2cc(Cl)cnc2-c2ccc(CO)nc2)cc1,2001.0
6,CHEMBL100422,30.0,COc1ccc(-c2cscc2-c2ccc(S(C)(=O)=O)cc2)cc1F,1995.0
7,CHEMBL101,284000.0,CCCCC1C(=O)N(c2ccccc2)N(c2ccccc2)C1=O,1995.0
8,CHEMBL101459,49000.0,CS(=O)(=O)c1ccc(-c2cc(CC#N)sc2-c2ccc(F)cc2)cc1,1996.0
9,CHEMBL101503,11000.0,CS(=O)(=O)c1ccc(-c2ccccc2-c2ccccc2)cc1,1996.0



CLEANED DATA STATISTICS:


,Metric,Value
0,Total Molecules,5371
1,Mean IC50 (nM),213232.8
2,Median IC50 (nM),1600.0
3,Min IC50 (nM),0.0
4,Max IC50 (nM),223342021.1



✓ Cleaned data saved to 'cox2_chembl_cleaned.csv'


Explanation of Output:
"Starting with 8,383 records, filtering reduced to 7,878 clean measurements. After handling duplicates, we have 3,641 unique molecules with IC50 values ranging from 0.1 nM to 450,000 nM. The cleaned data is saved for the next steps."